# The non-interactive contract

**Scenario:** a nightly job triages open incidents before stand-up. Three weeks of green runs. Then
a checkout outage sat unpaged for six hours, and the job that should have caught it exited 0 that
night, as it always did.

Two things change when nobody is attached to a run. Nothing can answer a question, so a confirmation
prompt is a phone ringing in an empty office. And the only thing your pipeline reads back is one
small integer.

## Mechanics

A headless run has five channels, and each one means something different to the job that called it.

| Channel | Who reads it | What it must carry |
|---|---|---|
| `stdin` | your step | nothing. Close it, or a prompt waits forever |
| `stdout` | the next step in the pipeline | the payload, and nothing else |
| `stderr` | a person reading the log later | narration, progress, warnings |
| exit code | the pipeline itself | 0 clean, 1 the run broke, 2 a real finding |
| wall clock | the runner | a bound you set, because there is no default |

The exit code is the whole interface. A pipeline never reads your sentences. It reads that integer
and decides whether the next job runs.

## The picture

![A headless step, its four channels and the exit code the pipeline branches on](images/non-interactive-contract.svg)

Everything a person would have watched on a screen leaves through one of those channels instead.

## The cost

```
burned  = runner minutes spent waiting before something kills the job
missed  = nights the step reported a finding x nights the job still exited 0
```

The first is billed by the minute. The second is billed to whoever gets paged instead.

## The failure

Start with the hang. Here is a step that asks for confirmation, started the way a runner starts it.

In [1]:
import subprocess
import sys
import time

ASKS = "input('Approve the rollback? [y/N] ')"

step = subprocess.Popen([sys.executable, "-c", ASKS], stdin=subprocess.PIPE,
                        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
started = time.monotonic()
try:
    step.wait(timeout=2.0)
    print("the step exited on its own")
except subprocess.TimeoutExpired:
    print(f"no exit after {time.monotonic() - started:.1f} seconds, and nobody is at the keyboard")
    step.kill()
    raise

no exit after 2.0 seconds, and nobody is at the keyboard


TimeoutExpired: Command '['/Users/param/learn/learnwithparam/lwp-repos/ai-engineering-vaults/.venv/bin/python3', '-c', "input('Approve the rollback? [y/N] ')"]' timed out after 2.0 seconds

Two seconds is the timeout, not the truth. Without that argument the step waits until the runner
kills the whole job, and the log says nothing about why.

The second failure is quieter. Here is the triage step.

In [2]:
from vault import get_client, load_env, model_for

load_env()
client = get_client("06-headless-automation/01-the-non-interactive-contract")

SYSTEM = ("You are the on-call triage step in a nightly job. "
          "Read the incident notes and say whether a human must be paged tonight.")
NOTES = ("INC-2214. Checkout error rate went from 0.2 percent to 11 percent at 02:14. "
         "It is still 9 percent. The deploy at 02:09 has not been rolled back. "
         "No one has acknowledged the alert.")


def triage_once():
    """One night of the nightly job. Returns whatever the model said."""
    reply = client.chat.completions.create(
        model=model_for("default"), max_tokens=200,
        messages=[{"role": "system", "content": SYSTEM},
                  {"role": "user", "content": NOTES}])
    return (reply.choices[0].message.content or "").strip()

The runner around it does what most first versions do. It checks that the step produced something.

In [3]:
nights = [triage_once() for _ in range(4)]


def naive_exit_code(report):
    """Green if the step said anything at all."""
    return 0 if report else 1


for report in nights:
    print(f"  exit {naive_exit_code(report)}   {report.splitlines()[0][:62]}")

paged = sum("paged" in report.lower() for report in nights)
print(f"\nthe step asked for a human on {paged} of {len(nights)} nights")
assert paged == 0, f"{paged} of {len(nights)} nights wanted a human and every run exited 0"

  exit 0   Yes, a human must be paged tonight. The checkout error rate ha
  exit 0   Yes, a human must be paged. The checkout error rate has signif
  exit 0   Yes, a human must be paged. The checkout error rate has increa
  exit 0   Yes, a human must be paged. The checkout error rate has signif

the step asked for a human on 4 of 4 nights


AssertionError: 4 of 4 nights wanted a human and every run exited 0

## The diagnosis

Two failures, one cause. Nothing in either run carried a decision.

**The hang.** `stdin` was a pipe with nobody on the other end. The step was written for a terminal,
where a person types y. In a runner there is no person, so the read never returns.

**The green build.** The runner asked the wrong question. It checked that the step produced output,
not that the output was clean. Look at the mechanics table: the exit code is the interface, and here
it was a side effect of the process not crashing.

A red finding under a green build is worse than no check at all, because the team now believes
something is watching.

## The fix

The fix closes two doors. The step cannot be asked a question, and it cannot run longer than you
allow.

In [4]:
def run_step(argv, seconds):
    """Run one step the way a pipeline has to. No keyboard, no open ended wait."""
    try:
        done = subprocess.run(argv, stdin=subprocess.DEVNULL, capture_output=True,
                              text=True, timeout=seconds)
        return done.returncode, done.stdout, done.stderr
    except subprocess.TimeoutExpired:
        return 124, "", f"killed after {seconds} seconds"

`stdin=subprocess.DEVNULL` is the whole hang fix. The read hits end of file at once, the step dies
with a real error, and the log carries the reason.

In [5]:
started = time.monotonic()
code, out, err = run_step([sys.executable, "-c", ASKS], seconds=10)
elapsed = time.monotonic() - started

print(f"before: still waiting after 2.0 seconds, and killed with nothing to show")
print(f"after : exit {code} in {elapsed:.2f} seconds")
print(f"reason: {err.strip().splitlines()[-1]}")

before: still waiting after 2.0 seconds, and killed with nothing to show
after : exit 1 in 0.04 seconds
reason: EOFError: EOF when reading a line


Now the second door. A verdict has to leave the step as a value, so the runner reads it rather than
guessing.

In [6]:
def verdict_of(report):
    """Read the one word the step promised. Loudly, or not at all."""
    last = report.strip().splitlines()[-1].strip()
    if last not in ("PAGE", "HOLD"):
        raise ValueError(f"no verdict on the last line: {last[:40]!r}")
    return last

That needs the step to make the promise, so the instruction asks for it. This is the weakest part of
the sub-module, deliberately. Sub-module two replaces the promise with something enforced.

In [7]:
STRICT = ("You are the on-call triage step in a nightly job. "
          "Read the incident notes. Answer in at most two sentences, then put "
          "one word on its own final line: PAGE or HOLD.")


def triage_strict():
    """The same call, with a contract on the last line."""
    reply = client.chat.completions.create(
        model=model_for("default"), max_tokens=200,
        messages=[{"role": "system", "content": STRICT},
                  {"role": "user", "content": NOTES}])
    return (reply.choices[0].message.content or "").strip()

Map the verdict to an exit code and the same four nights come out differently.

In [8]:
EXIT_FOR = {"HOLD": 0, "PAGE": 2}

strict_nights = [triage_strict() for _ in range(4)]
codes = [EXIT_FOR[verdict_of(report)] for report in strict_nights]

print(f"before: exit codes {[naive_exit_code(r) for r in nights]}")
print(f"after : exit codes {codes}")
print(f"nights the pipeline now stops on: {sum(code != 0 for code in codes)} of {len(codes)}")

before: exit codes [0, 0, 0, 0]
after : exit codes [2, 2, 2, 2]
nights the pipeline now stops on: 4 of 4


## The gate

The check that stops this coming back needs no model, so it runs on every commit in a second.

In [9]:
def test_a_prompt_cannot_stall_the_pipeline():
    code, _, err = run_step([sys.executable, "-c", ASKS], seconds=5)
    assert code != 0, "a step that asks a question must fail, not pass"
    assert "EOFError" in err, f"expected an end of file error, got {err[-60:]!r}"


test_a_prompt_cannot_stall_the_pipeline()
print("gate holds: a step that waits for a person fails fast instead of hanging")

gate holds: a step that waits for a person fails fast instead of hanging


Drop `stdin=subprocess.DEVNULL` from `run_step` and the step waits the full five seconds before
something kills it. The test then fails on the second assertion, because the log holds a timeout
notice instead of a reason.

### Enterprise exploration

- Runners charge by the minute. What does one hung job cost before anything kills it, and who sees
  that number?
- What breaks in your pipeline the day somebody collapses exit 1 and exit 2 into one code?
- An auditor asks which nights the step reported a finding. Nothing records that. What is the
  compliance cost of answering late?
- At what scale does a per-step timeout stop being enough, and what would you measure to know?

### Key takeaways

- A prompt in a headless run is a hang, not a question.
- Close `stdin` and the hang becomes a fast, loud failure with a reason in the log.
- The exit code is the interface. Wire it to the finding, not to whether the process survived.